In [ ]:
from nhlpy import NHLClient

# Basic usage
client = NHLClient()

# Get all teams
teams = client.teams.teams()
print(teams)

: 

In [ ]:
roster = client.teams.team_roster(team_abbr="SJS", season="20252026")
print(f"Forwards: {len(roster['forwards'])}")
print(f"Defensemen: {len(roster['defensemen'])}")
print(f"Goalies: {len(roster['goalies'])}")
print(roster['forwards'])
# Macklin ID: 8484801
sharksfwds = []
for each in roster['forwards']:
  fname = str(each['firstName']['default'])
  lname = str(each['lastName']['default'])
  id = each['id']
  fullname = fname + ' ' + lname
  playerdict = {'id': id, 'name' : fullname}
  sharksfwds.append(playerdict)

print(sharksfwds)


: 

In [ ]:
# Get current season skater stats
stats = client.stats.skater_stats_summary(
    start_season="20252026", 
    end_season="20252026"
)

# Show top 5 scorers
for i, player in enumerate(stats[:5]):
    print(f"{i+1}. {player['skaterFullName']}: {player['goals']} goals")
    print(player)

: 

In [6]:
import pandas as pd
sharkies = pd.DataFrame(sharksfwds)
print(sharkies)
macky = sharkies[sharkies['name'] == 'Macklin Celebrini']['id'][0]
print(macky)

         id                name
0   8484801   Macklin Celebrini
1   8480848       Ty Dellandrea
2   8482667      William Eklund
3   8478874       Adam Gaudette
4   8476624     Barclay Goodrow
5   8484911         Collin Graf
6   8480798    Philipp Kurashev
7   8485402        Michael Misa
8   8482859      Zack Ostapchuk
9   8471817         Ryan Reaves
10  8483630       Pavol Regenda
11  8475784        Jeff Skinner
12  8484227          Will Smith
13  8475726       Tyler Toffoli
14  8477505  Alexander Wennberg
8484801


In [51]:
import httpx
sharksskatedist = []
for index, row in sharkies.iterrows():
  playerid = row['id']
  
  try:
    skating = client.edge.skater_skating_distance_detail(player_id=playerid, season='20252026')
    skating = skating['skatingDistanceLast10']

    homedist = 0
    awaydist = 0
    for each in skating:
      if each['playerOnHomeTeam'] == True:
        homedist = homedist + each['distanceSkatedAll']['imperial']
      else :
        awaydist = awaydist + each['distanceSkatedAll']['imperial']
 
  except httpx.HTTPStatusError:
        homedist = None
        awaydist = None

  except Exception:
      homedist = None
      awaydist = None
  
  playerdict = {'name' : row['name'], 'homeskatedist' : homedist, 'awayskatedist' : awaydist}
  sharksskatedist.append(playerdict)

sjsskate = pd.DataFrame(sharksskatedist)
display(sjsskate)  

,name,homeskatedist,awayskatedist
0,Macklin Celebrini,17.2001,17.0670
1,Ty Dellandrea,12.2471,12.7033
2,William Eklund,15.6698,15.7553
3,Adam Gaudette,9.7699,10.8479
4,Barclay Goodrow,10.4436,8.4036
5,Collin Graf,11.8850,13.0398
6,Philipp Kurashev,11.9009,12.8026
7,Michael Misa,7.8499,5.7856
8,Zack Ostapchuk,8.0864,4.6386
9,Ryan Reaves,6.9082,6.6573


In [7]:
game_log = client.stats.player_game_log(
    player_id="8484227", 
    season_id="20252026", 
    game_type=2  # Regular season
)

In [10]:
for game in game_log:
  game['commonName'] = game['commonName']['default']
  game['opponentCommonName'] = game['opponentCommonName']['default']
smittygames = pd.DataFrame(game_log)
display(smittygames)

,gameId,teamAbbrev,homeRoadFlag,gameDate,goals,assists,commonName,opponentCommonName,points,plusMinus,...,powerPlayPoints,gameWinningGoals,otGoals,shots,shifts,shorthandedGoals,shorthandedPoints,toi,opponentAbbrev,pim
0,2025020410,SJS,H,2025-12-01,2,1,Sharks,Mammoth,3,2,...,0,1,0,5,20,0,0,19:03,UTA,0
1,2025020400,SJS,R,2025-11-29,2,0,Sharks,Golden Knights,2,0,...,1,0,0,4,23,0,0,19:53,VGK,0
2,2025020385,SJS,H,2025-11-28,1,0,Sharks,Canucks,1,-1,...,1,0,0,3,20,0,0,18:24,VAN,2
3,2025020371,SJS,R,2025-11-26,0,0,Sharks,Avalanche,0,-2,...,0,0,0,4,21,0,0,18:06,COL,0
4,2025020351,SJS,H,2025-11-23,0,0,Sharks,Bruins,0,0,...,0,0,0,7,20,0,0,15:25,BOS,0
5,2025020342,SJS,H,2025-11-22,0,1,Sharks,Senators,1,0,...,1,0,0,2,21,0,0,21:27,OTT,0
6,2025020330,SJS,H,2025-11-20,0,1,Sharks,Kings,1,0,...,0,0,0,1,20,0,0,18:44,LAK,0
7,2025020314,SJS,H,2025-11-18,0,2,Sharks,Mammoth,2,0,...,1,0,0,0,19,0,0,17:23,UTA,0
8,2025020295,SJS,R,2025-11-15,0,0,Sharks,Kraken,0,-3,...,0,0,0,5,19,0,0,19:04,SEA,0
9,2025020276,SJS,R,2025-11-13,0,0,Sharks,Flames,0,-2,...,0,0,0,3,16,0,0,14:27,CGY,0


In [ ]:
smittylogs = []
for game in game_log :
  gamedate = game['gameDate']
  goals = game['goals']
  assists = game['assists']
  shots = game['shots']
  toi = game['toi']
  

[{'gameId': 2025020410,
  'teamAbbrev': 'SJS',
  'homeRoadFlag': 'H',
  'gameDate': '2025-12-01',
  'goals': 2,
  'assists': 1,
  'commonName': {'default': 'Sharks'},
  'opponentCommonName': {'default': 'Mammoth'},
  'points': 3,
  'plusMinus': 2,
  'powerPlayGoals': 0,
  'powerPlayPoints': 0,
  'gameWinningGoals': 1,
  'otGoals': 0,
  'shots': 5,
  'shifts': 20,
  'shorthandedGoals': 0,
  'shorthandedPoints': 0,
  'toi': '19:03',
  'opponentAbbrev': 'UTA',
  'pim': 0},
 {'gameId': 2025020400,
  'teamAbbrev': 'SJS',
  'homeRoadFlag': 'R',
  'gameDate': '2025-11-29',
  'goals': 2,
  'assists': 0,
  'commonName': {'default': 'Sharks'},
  'opponentCommonName': {'default': 'Golden Knights'},
  'points': 2,
  'plusMinus': 0,
  'powerPlayGoals': 1,
  'powerPlayPoints': 1,
  'gameWinningGoals': 0,
  'otGoals': 0,
  'shots': 4,
  'shifts': 23,
  'shorthandedGoals': 0,
  'shorthandedPoints': 0,
  'toi': '19:53',
  'opponentAbbrev': 'VGK',
  'pim': 0},
 {'gameId': 2025020385,
  'teamAbbrev': 'SJ

In [27]:
client.edge.skater_detail(player_id='8484801', season='20242025')

{'player': {'id': 8484801,
  'firstName': {'default': 'Macklin'},
  'lastName': {'default': 'Celebrini'},
  'birthDate': '2006-06-13',
  'shootsCatches': 'L',
  'sweaterNumber': 71,
  'position': 'C',
  'slug': 'macklin-celebrini-8484801',
  'headshot': 'https://assets.nhle.com/mugs/nhl/20242025/SJS/8484801.png',
  'goals': 25,
  'assists': 38,
  'points': 63,
  'gamesPlayed': 70,
  'team': {'commonName': {'default': 'Sharks'},
   'placeNameWithPreposition': {'default': 'San Jose', 'fr': 'de San Jose'},
   'abbrev': 'SJS',
   'teamLogo': {'light': 'https://assets.nhle.com/logos/nhl/svg/SJS_light.svg',
    'dark': 'https://assets.nhle.com/logos/nhl/svg/SJS_dark.svg'}}},
 'seasonsWithEdgeStats': [{'id': 20242025, 'gameTypes': [2]},
  {'id': 20252026, 'gameTypes': [2]}],
 'topShotSpeed': {'imperial': 89.64,
  'metric': 144.2616,
  'percentile': 0.7677,
  'leagueAvg': {'imperial': 83.9219, 'metric': 135.0592},
  'overlay': {'player': {'firstName': {'default': 'Macklin'},
    'lastName': {'

In [ ]:
# Get current standings
standings = client.standings.league_standings()
print(standings)

# Get today's games
games = client.schedule.daily_schedule()

{'wildCardIndicator': True, 'standingsDateTimeUtc': '2025-12-01T18:39:45Z', 'standings': [{'conferenceAbbrev': 'W', 'conferenceHomeSequence': 1, 'conferenceL10Sequence': 1, 'conferenceName': 'Western', 'conferenceRoadSequence': 3, 'conferenceSequence': 1, 'date': '2025-12-01', 'divisionAbbrev': 'C', 'divisionHomeSequence': 1, 'divisionL10Sequence': 1, 'divisionName': 'Central', 'divisionRoadSequence': 2, 'divisionSequence': 1, 'gameTypeId': 2, 'gamesPlayed': 25, 'goalDifferential': 48, 'goalDifferentialPctg': 1.92, 'goalAgainst': 55, 'goalFor': 103, 'goalsForPctg': 4.12, 'homeGamesPlayed': 12, 'homeGoalDifferential': 30, 'homeGoalsAgainst': 28, 'homeGoalsFor': 58, 'homeLosses': 0, 'homeOtLosses': 2, 'homePoints': 22, 'homeRegulationPlusOtWins': 10, 'homeRegulationWins': 10, 'homeTies': 0, 'homeWins': 10, 'l10GamesPlayed': 10, 'l10GoalDifferential': 27, 'l10GoalsAgainst': 17, 'l10GoalsFor': 44, 'l10Losses': 0, 'l10OtLosses': 1, 'l10Points': 19, 'l10RegulationPlusOtWins': 9, 'l10Regulati

: 

In [15]:
print(standings['standings'][0].keys())

dict_keys(['conferenceAbbrev', 'conferenceHomeSequence', 'conferenceL10Sequence', 'conferenceName', 'conferenceRoadSequence', 'conferenceSequence', 'date', 'divisionAbbrev', 'divisionHomeSequence', 'divisionL10Sequence', 'divisionName', 'divisionRoadSequence', 'divisionSequence', 'gameTypeId', 'gamesPlayed', 'goalDifferential', 'goalDifferentialPctg', 'goalAgainst', 'goalFor', 'goalsForPctg', 'homeGamesPlayed', 'homeGoalDifferential', 'homeGoalsAgainst', 'homeGoalsFor', 'homeLosses', 'homeOtLosses', 'homePoints', 'homeRegulationPlusOtWins', 'homeRegulationWins', 'homeTies', 'homeWins', 'l10GamesPlayed', 'l10GoalDifferential', 'l10GoalsAgainst', 'l10GoalsFor', 'l10Losses', 'l10OtLosses', 'l10Points', 'l10RegulationPlusOtWins', 'l10RegulationWins', 'l10Ties', 'l10Wins', 'leagueHomeSequence', 'leagueL10Sequence', 'leagueRoadSequence', 'leagueSequence', 'losses', 'otLosses', 'placeName', 'pointPctg', 'points', 'regulationPlusOtWinPctg', 'regulationPlusOtWins', 'regulationWinPctg', 'regulat

In [ ]:
import pandas as pd
print(standings['standings'])
for each in standings['standings']:
  print(each)


[{'conferenceAbbrev': 'W', 'conferenceHomeSequence': 1, 'conferenceL10Sequence': 1, 'conferenceName': 'Western', 'conferenceRoadSequence': 3, 'conferenceSequence': 1, 'date': '2025-12-01', 'divisionAbbrev': 'C', 'divisionHomeSequence': 1, 'divisionL10Sequence': 1, 'divisionName': 'Central', 'divisionRoadSequence': 2, 'divisionSequence': 1, 'gameTypeId': 2, 'gamesPlayed': 25, 'goalDifferential': 48, 'goalDifferentialPctg': 1.92, 'goalAgainst': 55, 'goalFor': 103, 'goalsForPctg': 4.12, 'homeGamesPlayed': 12, 'homeGoalDifferential': 30, 'homeGoalsAgainst': 28, 'homeGoalsFor': 58, 'homeLosses': 0, 'homeOtLosses': 2, 'homePoints': 22, 'homeRegulationPlusOtWins': 10, 'homeRegulationWins': 10, 'homeTies': 0, 'homeWins': 10, 'l10GamesPlayed': 10, 'l10GoalDifferential': 27, 'l10GoalsAgainst': 17, 'l10GoalsFor': 44, 'l10Losses': 0, 'l10OtLosses': 1, 'l10Points': 19, 'l10RegulationPlusOtWins': 9, 'l10RegulationWins': 8, 'l10Ties': 0, 'l10Wins': 9, 'leagueHomeSequence': 1, 'leagueL10Sequence': 1, 

: 

: 